# Snow segmentation training

This notebook trains state-of-the-art semantic segmentation models on the curated Sentinel-2 snow dataset. It reuses the encoders/decoders and PyTorch Lightning model definitions from `2_train_state_of_the_art` while adding 5-fold cross-validation, manifest-driven dataloaders, and live visualizations of training progress.


In [1]:
import os
import random
from pathlib import Path
from typing import Iterable, Optional, Sequence, Tuple, Union

import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import rasterio
from rasterio.enums import Resampling
import torch
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import InterpolationMode
from torch.optim import lr_scheduler

from my_code.model import MyModel

C:\Users\Domen\anaconda3\lib\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision('medium')
torch.backends.cudnn.benchmark = True

Global seed set to 42


In [3]:
CONFIG = {
    "num_folds": 5,
    "max_epochs": 35,
    "batch_size": 8,
    "num_workers": 0, #2,
    "learning_rate": 6e-4,
    "weight_decay": 1e-2,
    "optimizer": "adamw",
    "scheduler": "polynomial",
    "loss_function": "CrossEntropyLoss",  # or "FocalLoss"
    "architecture": "GASSL-basic", #"Unet", "GASSL-basic"
    "image_size": 512,
    "mask_size": 1024,
    "rgb_indexes": (4, 3, 2),
    "use_all_bands": False,
    "band_mode": "rgb",  # set to "rgb" or "all" to override use_all_bands
    "augment": True,
    "stats_sample_limit": 64,
    "recompute_stats_per_fold": False,
    "log_every_n_steps": 20,
    "grad_clip": 1.0,
    "precision": "16",
    "monitor_metric": "valid_mIOU",
}

NOTEBOOK_CWD = Path(os.getcwd()).resolve()
if (NOTEBOOK_CWD / "2_train_state_of_the_art").exists():
    TRAINING_ROOT = NOTEBOOK_CWD / "2_train_state_of_the_art"
else:
    TRAINING_ROOT = NOTEBOOK_CWD
RUNS_DIR = TRAINING_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

BAND_MODE = CONFIG.get("band_mode")
if BAND_MODE is None:
    BAND_MODE = "all_spectral_bands" if CONFIG["use_all_bands"] else "rgb"

DATASET_ROOT_CANDIDATES = [
    Path("1_download_data_and_create_dataset/dataset"),
    Path("../1_download_data_and_create_dataset/dataset"),
    Path("../../1_download_data_and_create_dataset/dataset"),
]

def find_dataset_root() -> Path:
    for candidate in DATASET_ROOT_CANDIDATES:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate dataset directory. Please update DATASET_ROOT_CANDIDATES.")

DATASET_ROOT = find_dataset_root()
MANIFEST_PATH = DATASET_ROOT / "manifest.csv"
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Loaded {len(manifest)} tiles from {MANIFEST_PATH}")


Loaded 8299 tiles from D:\analiza_snezisc2\1_download_data_and_create_dataset\dataset\manifest.csv


In [4]:
MIN_RAW_VARIANCE = 1e-4

def resolve_path(relative_path: str) -> Path:
    normalized = relative_path.replace('\\', '/')
    return DATASET_ROOT / Path(normalized)


def _normalize_out_size(out_size: Union[int, Tuple[int, int], None]) -> Optional[Tuple[int, int]]:
    if out_size is None:
        return None
    if isinstance(out_size, int):
        return (out_size, out_size)
    return tuple(out_size)


def read_multispectral_image(
    path: Path,
    band_indexes: Sequence[int],
    out_size: Union[int, Tuple[int, int], None] = None,
) -> np.ndarray:
    with rasterio.open(path) as src:
        if not band_indexes:
            selected = list(range(1, src.count + 1))
        else:
            selected = [idx for idx in band_indexes if idx <= src.count]
            if not selected:
                selected = [1]
        target_size = _normalize_out_size(out_size)
        if target_size is not None:
            data = src.read(
                selected,
                out_shape=(len(selected), target_size[0], target_size[1]),
                resampling=Resampling.bilinear,
            ).astype(np.float32)
        else:
            data = src.read(selected).astype(np.float32)
    data = np.moveaxis(data, 0, -1)
    data = np.clip(data, 0, None)
    if not np.isfinite(data).all():
        raise ValueError(f"Non-finite values in {path}")
    if data.ptp() < MIN_RAW_VARIANCE:
        raise ValueError(f"Image {path} lacks contrast")
    max_val = data.max()
    if max_val > 0:
        data /= max_val
    return data


def read_mask(
    path: Path,
    out_size: Union[int, Tuple[int, int], None] = None,
) -> np.ndarray:
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.int64)
    target_size = _normalize_out_size(out_size)
    if target_size is not None and (data.shape[0] != target_size[0] or data.shape[1] != target_size[1]):
        data = cv2.resize(
            data,
            dsize=(target_size[1], target_size[0]),
            interpolation=cv2.INTER_NEAREST,
        )
    return data


def joint_random_flip(
    image: torch.Tensor, mask: torch.Tensor, p: float = 0.5
) -> Tuple[torch.Tensor, torch.Tensor]:
    if random.random() < p:
        image = TF.hflip(image)
        mask = torch.flip(mask, dims=[-1])
    if random.random() < p:
        image = TF.vflip(image)
        mask = torch.flip(mask, dims=[-2])
    return image, mask


class SnowSegmentationDataset(Dataset):
    def __init__(
        self,
        manifest_df: pd.DataFrame,
        indices: Sequence[int],
        band_indexes: Sequence[int],
        image_size: Optional[int],
        mask_size: Optional[int],
        augment: bool = False,
    ):
        self.manifest_df = manifest_df.reset_index(drop=True)
        self.indices = list(indices)
        self.band_indexes = tuple(band_indexes)
        self.image_size = image_size
        self.mask_size = mask_size
        self.augment = augment

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        entry = self.manifest_df.iloc[self.indices[idx]]
        image = read_multispectral_image(
            resolve_path(entry['sentinel_path']),
            self.band_indexes,
            out_size=self.image_size,
        )
        mask = read_mask(
            resolve_path(entry['snow_mask_path']),
            out_size=self.mask_size,
        )
        image_tensor = torch.from_numpy(np.moveaxis(image, -1, 0)).float()
        mask_tensor = torch.from_numpy(mask).long()
        if self.augment:
            image_tensor, mask_tensor = joint_random_flip(image_tensor, mask_tensor)
        return image_tensor, mask_tensor


def compute_channel_stats(
    manifest_df: pd.DataFrame,
    indices: Sequence[int],
    band_indexes: Sequence[int],
    sample_limit: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = random.Random(42)
    order = list(indices)
    rng.shuffle(order)
    if sample_limit is not None:
        order = order[:sample_limit]
    sums = None
    sums_sq = None
    total = 0
    for idx in order:
        entry = manifest_df.iloc[idx]
        img = read_multispectral_image(resolve_path(entry['sentinel_path']), band_indexes)
        pixels = img.reshape(-1, img.shape[-1])
        if sums is None:
            sums = np.zeros(pixels.shape[1], dtype=np.float64)
            sums_sq = np.zeros_like(sums)
        sums += pixels.sum(axis=0)
        sums_sq += np.square(pixels).sum(axis=0)
        total += pixels.shape[0]
    means = sums / total
    stds = np.sqrt(np.maximum(sums_sq / total - means ** 2, 1e-12))
    return means, stds


In [5]:
def determine_band_indexes() -> Tuple[int, ...]:
    sample_entry = manifest.iloc[0]
    with rasterio.open(resolve_path(sample_entry['sentinel_path'])) as src:
        use_all = CONFIG["use_all_bands"]
        if CONFIG.get("band_mode") == "all":
            use_all = True
        elif CONFIG.get("band_mode") == "rgb":
            use_all = False
        if use_all:
            return tuple(range(1, src.count + 1))
        selected = [idx for idx in CONFIG["rgb_indexes"] if idx <= src.count]
        if not selected:
            selected = [1]
        return tuple(selected)

SELECTED_BANDS = determine_band_indexes()
print(f"Using bands: {SELECTED_BANDS} (total {len(SELECTED_BANDS)})")

Using bands: (4, 3, 2) (total 3)


In [6]:
class SnowSegLightningModule(MyModel):
    def __init__(
        self,
        arch: str,
        in_channels: int,
        train_means,
        train_stds,
        loss_function: str,
        optimizer_name: str,
        learning_rate: float,
        weight_decay: float,
        scheduler_name: str,
        n_iters_training: int
    ):
        super().__init__(
            arch,
            in_channels,
            out_classes=2,
            train_channel_means=train_means,
            train_channel_stds=train_stds,
            loss_function=loss_function,
            n_iters_training=n_iters_training,
        )
        self.optimizer_name = optimizer_name.lower()
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.scheduler_name = scheduler_name.lower()

    def configure_optimizers(self):
        if self.optimizer_name == "adam":
            optimizer = torch.optim.Adam(
                self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay
            )
        elif self.optimizer_name == "sgd":
            optimizer = torch.optim.SGD(
                self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=self.weight_decay
            )
        else:
            optimizer = torch.optim.AdamW(
                self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay
            )
        if self.scheduler_name == "polynomial":
            scheduler = lr_scheduler.PolynomialLR(
                optimizer, total_iters=max(1, int(self.n_iters_training))
            )
            return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}
        elif self.scheduler_name == "cosine":
            scheduler = lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=max(1, int(self.n_iters_training))
            )
            return {"optimizer": optimizer, "lr_scheduler": scheduler}
        return optimizer

class PredictionPreviewCallback(pl.callbacks.Callback):
    def __init__(self, fold_dir: Path, dataset: Dataset, num_examples: int = 3):
        self.fold_dir = Path(fold_dir)
        self.dataset = dataset
        self.num_examples = num_examples
        self.output_dir = self.fold_dir / "previews"
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def on_validation_epoch_end(self, trainer, pl_module):
        if len(self.dataset) == 0:
            return
        pl_module.eval()
        device = pl_module.device
        rng = np.random.default_rng(seed=trainer.current_epoch)
        sample_count = min(self.num_examples, len(self.dataset))
        indices = rng.choice(len(self.dataset), size=sample_count, replace=False)
        for idx in indices:
            image_tensor, mask_tensor = self.dataset[idx]
            image_np = image_tensor.cpu().numpy()
            input_batch = image_tensor.unsqueeze(0).to(device)
            with torch.no_grad():
                logits = pl_module(input_batch)
            pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            target_mask = mask_tensor.cpu().numpy()

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            show_image = image_np[:3] if image_np.shape[0] >= 3 else image_np
            axes[0].imshow(np.moveaxis(show_image, 0, -1))
            axes[0].set_title("Input")
            axes[1].imshow(target_mask, cmap='gray')
            axes[1].set_title("Target")
            axes[2].imshow(pred_mask, cmap='gray')
            axes[2].set_title("Prediction")
            for ax in axes:
                ax.axis('off')
            outfile = self.output_dir / f"epoch_{trainer.current_epoch:03d}_sample_{idx}.png"
            fig.tight_layout()
            fig.savefig(outfile)
            plt.close(fig)
class ProgressFileLogger(pl.callbacks.Callback):
    def __init__(self, log_path: Path):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.log_path.write_text("")

    def _write_line(self, text: str) -> None:
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(text + "\n")

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx) -> None:
        num_batches = trainer.num_training_batches or 1
        current = trainer.fit_loop.epoch_loop._batches_that_stepped
        percent = 100.0 * current / num_batches
        metrics = {}
        progress_cb = getattr(trainer, "progress_bar_callback", None)
        if progress_cb and hasattr(progress_cb, "get_metrics"):
            metrics = progress_cb.get_metrics(trainer, pl_module)
        elif trainer.progress_bar_dict:
            metrics = trainer.progress_bar_dict
        metric_str = ", ".join(f"{k}={v}" for k, v in metrics.items())
        line = f"Epoch {trainer.current_epoch}: {percent:.1f}% | batch {current}/{num_batches} | {metric_str}"
        self._write_line(line)

    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
        self._write_line(f"Validation epoch {trainer.current_epoch}: {metric_str}")


class HistoryCallback(pl.callbacks.Callback):
    def __init__(self, fold_id: int):
        self.fold_id = fold_id
        self.history = []

    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        record = {"fold": self.fold_id, "epoch": int(trainer.current_epoch)}
        for key in ["train_loss", "valid_loss", "train_mIOU", "valid_mIOU"]:
            value = metrics.get(key)
            if value is not None:
                record[key] = float(value.detach().cpu())
        self.history.append(record)


In [ ]:
kf = KFold(n_splits=CONFIG["num_folds"], shuffle=True, random_state=42)
all_histories = []
fold_summaries = []
base_means, base_stds = compute_channel_stats(
    manifest, range(len(manifest)), SELECTED_BANDS, sample_limit=CONFIG["stats_sample_limit"]
)

stats_payload = {
    "rgb_indexes": list(SELECTED_BANDS),
    "sample_limit": CONFIG["stats_sample_limit"],
    "means": np.asarray(base_means).tolist(),
    "stds": np.asarray(base_stds).tolist(),
}
stats_path = RUNS_DIR / "channel_stats.json"
stats_path.write_text(json.dumps(stats_payload, indent=2))
print(f"Saved global channel statistics to {stats_path}")

for fold_id, (train_idx, val_idx) in enumerate(kf.split(manifest), start=1):
    print(f"==== Fold {fold_id}/{CONFIG['num_folds']} ====")
    fold_root = RUNS_DIR / f"fold_{fold_id}"
    fold_dir = fold_root / BAND_MODE / CONFIG["architecture"] / CONFIG["loss_function"]
    fold_dir.mkdir(parents=True, exist_ok=True)
    log_path = fold_dir / "training.log"

    log_path = fold_dir / "training.log"


    train_dataset = SnowSegmentationDataset(
        manifest, train_idx, SELECTED_BANDS, CONFIG["image_size"], CONFIG["mask_size"], augment=CONFIG["augment"]
    )
    val_dataset = SnowSegmentationDataset(
        manifest, val_idx, SELECTED_BANDS, CONFIG["image_size"], CONFIG["mask_size"], augment=False
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        persistent_workers=CONFIG["num_workers"] > 0,
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        persistent_workers=CONFIG["num_workers"] > 0,
        drop_last=True
    )

    if CONFIG["recompute_stats_per_fold"]:
        means, stds = compute_channel_stats(
            manifest, train_idx, SELECTED_BANDS, sample_limit=CONFIG["stats_sample_limit"]
        )
    else:
        means, stds = base_means, base_stds
    print("Channel means:", means, "stds:", stds)

    fold_stats_payload = {
        "fold": fold_id,
        "rgb_indexes": list(SELECTED_BANDS),
        "means": np.asarray(means).tolist(),
        "stds": np.asarray(stds).tolist(),
    }
    (fold_dir / "channel_stats.json").write_text(json.dumps(fold_stats_payload, indent=2))

    n_iters = max(1, len(train_loader) * CONFIG["max_epochs"])
    model = SnowSegLightningModule(
        arch=CONFIG["architecture"],
        in_channels=len(SELECTED_BANDS),
        train_means=means,
        train_stds=stds,
        loss_function=CONFIG["loss_function"],
        optimizer_name=CONFIG["optimizer"],
        learning_rate=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
        scheduler_name=CONFIG["scheduler"],
        n_iters_training=n_iters,
    )

    checkpoint_cb = pl.callbacks.ModelCheckpoint(
        dirpath=str(fold_dir / "checkpoints"),
        filename="snow-best",
        monitor=CONFIG["monitor_metric"],
        mode="max",
        save_top_k=1,
    )
    history_cb = HistoryCallback(fold_id)
    preview_cb = PredictionPreviewCallback(fold_dir, val_dataset, num_examples=5)
    progress_logger = ProgressFileLogger(log_path)

    trainer = pl.Trainer(
        logger=pl.loggers.CSVLogger(save_dir=str(fold_dir), name="lightning", version=trainer.current_epoch if False else None),
        max_epochs=CONFIG["max_epochs"],
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision=CONFIG["precision"] if torch.cuda.is_available() else 32,
        gradient_clip_val=CONFIG["grad_clip"],
        default_root_dir=str(fold_dir),
        log_every_n_steps=CONFIG["log_every_n_steps"],
        callbacks=[history_cb, checkpoint_cb, preview_cb, progress_logger],
        enable_progress_bar=True,
        num_sanity_val_steps=0,
    )
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    all_histories.extend(history_cb.history)
    best_score = checkpoint_cb.best_model_score
    fold_summaries.append(
        {
            "fold": fold_id,
            "best_metric": float(best_score) if best_score is not None else None,
            "best_model_path": checkpoint_cb.best_model_path,
        }
    )

fold_summary_df = pd.DataFrame(fold_summaries)
display(fold_summary_df)


Saved global channel statistics to D:\analiza_snezisc2\3_train_state_of_the_art\runs\channel_stats.json
==== Fold 1/5 ====
Channel means: [0.43938719 0.4088533  0.45922646] stds: [0.26054574 0.19455334 0.20731123]
=> loading checkpoint './methods/GASSL/moco_fmow/weights/moco.pth.tar'
=> loaded MoCo‐pretrained weights from './methods/GASSL/moco_fmow/weights/moco.pth.tar'


Using 16bit None Automatic Mixed Precision (AMP)
C:\Users\Domen\AppData\Roaming\Python\Python310\site-packages\pytorch_lightning\plugins\precision\native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


318 tensor(s) updated after loading weights...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                    | Type                 | Params
-----------------------------------------------------------------
0 | scaling_transform       | Compose              | 0     
1 | encoder                 | MoCoResNet50Backbone | 23.5 M
2 | decoder                 | UPerNetDecoder       | 13.8 M
3 | segmentation_head       | SegmentationHead     | 514   
4 | loss_fn                 | SoftCrossEntropyLoss | 0     
5 | augmentation_transform  | Compose              | 0     
6 | normalization_transform | Compose              | 0     
-----------------------------------------------------------------
37.3 M    Trainable params
0         Non-trainable params
37.3 M    Total params
74.570    Total estimated model params size (MB)
C:\Users\Domen\AppData\Roaming\Python\Python310\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may b

Training: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
history_df = pd.DataFrame(all_histories)
if history_df.empty:
    print("No history captured yet.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
    for fold_id, fold_hist in history_df.groupby('fold'):
        if 'train_loss' in fold_hist:
            axes[0].plot(
                fold_hist['epoch'], fold_hist.get('train_loss', np.nan), label=f'train fold {fold_id}'
            )
        if 'valid_loss' in fold_hist:
            axes[0].plot(
                fold_hist['epoch'], fold_hist.get('valid_loss', np.nan), linestyle='--', label=f'val fold {fold_id}'
            )
        if 'train_mIOU' in fold_hist:
            axes[1].plot(
                fold_hist['epoch'], fold_hist.get('train_mIOU', np.nan), label=f'train fold {fold_id}'
            )
        if 'valid_mIOU' in fold_hist:
            axes[1].plot(
                fold_hist['epoch'], fold_hist.get('valid_mIOU', np.nan), linestyle='--', label=f'val fold {fold_id}'
            )
    axes[0].set_title('Loss per epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[1].set_title('mIoU per epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('mIoU')
    for ax in axes:
        ax.grid(True, linestyle=':')
        ax.legend()
    plt.tight_layout()
    Path("./runs/k_fold_validation_summary/" + BAND_MODE + "/" + CONFIG["architecture"] + "/" + CONFIG["loss_function"]).mkdir(parents=True, exist_ok=True)
    plt.savefig("./runs/k_fold_validation_summary/" + BAND_MODE + "/" + CONFIG["architecture"] + "/" + CONFIG["loss_function"] + "/history.png", dpi=300)